Deals with conflicting libraries

For faster results run everything using a T4 GPU

Make sure to add the Nist file to the files section

In [1]:
# 1. Force uninstall the conflicting libraries
!pip uninstall -y numpy scipy scikit-learn pandas

# 2. Reinstall them with strict version matching ("The Safe Stack")
# We use scipy<1.13 because newer versions caused the '_promote' error you are seeing.
!pip install "numpy<2.0" "scipy<1.13" "scikit-learn" "pandas" rich rich-theme-manager qdrant-client sentence-transformers accelerate transformers

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 71.7 MB/s eta 0:0

Imports libraries and defines the theme for the table

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# --- STEP 2: VISUAL SETUP ---
from rich.style import Style
from rich_theme_manager import Theme, ThemeManager
import pathlib
import pandas as pd
import warnings
import sys

# --- STEP 4: SETUP VECTOR DB ---
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Define Theme
THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),
            "repr.tag_name": "dim cyan",
            "repr.call": "bright_yellow",
            "repr.str": "bright_green",
            "repr.number": "bright_red",
            "repr.none": "dim white",
            "repr.attrib_name": Style(color="#e87d3e", bold=True),
            "repr.attrib_value": "bright_blue",
            "default": "bright_white on black"
        },
    )
]
theme_dir = pathlib.Path("themes").expanduser()
theme_dir.mkdir(parents=True, exist_ok=True)
theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
console = Console(theme=theme_manager.get("dark"))

Initial Setup and Loads Dataset

In [2]:
# --- SETUP (Fast Reload) ---
console = Console()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-3B-Instruct"

# --- STEP 3: LOAD DATA ---
warnings.filterwarnings('ignore')
console.print("[bold green]Loading NIST Data...[/bold green]")

try:
    df = pd.read_csv('NIST_SP-800-53.csv')
    df['combined_text'] = (
        "Control ID: " + df['identifier'].astype(str) + "; " +
        "Title: " + df['name'].astype(str) + "; " +
        "Control Text: " + df['control_text'].fillna('').astype(str) + "; " +
        "Discussion: " + df['discussion'].fillna('').astype(str)
    )
    data = df.to_dict('records')
    console.print(f"[dim]Loaded {len(data)} rows successfully.[/dim]")
except FileNotFoundError:
    console.print("[bold red]CRITICAL ERROR: CSV file not found![/bold red]")
    console.print("[yellow]Please re-upload 'NIST_SP-800-53.csv' to the Files panel.[/yellow]")
    data = []

# 1. Load Resources (Only if not already loaded to save time)
if 'qdrant' not in globals():
    console.print("[bold yellow]Reloading Database connection...[/bold yellow]")
    qdrant = QdrantClient(":memory:")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    # Re-index data if needed (Assuming 'data' variable exists from previous run)
    if 'data' in globals() and data:
        qdrant.recreate_collection(
            collection_name="nist_controls",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        qdrant.upload_points(
            collection_name="nist_controls",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(data)
            ]
        )

if 'model' not in globals():
    console.print(f"[bold yellow]Loading Model ({device})...[/bold yellow]")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)

Loading NIST Data...

Loaded 1189 rows successfully.

Reloading Database connection...

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Chat Loop

In [3]:
# --- MAIN CHAT LOOP ---
console.print("\n[bold green]✅ System Ready! Type 'exit' or 'quit' to stop.[/bold green]")

while True:
    # 1. Get Question
    console.print("\n[bold cyan]Your Question:[/bold cyan]")
    user_prompt = input(">>> ")

    if user_prompt.lower() in ["exit", "quit", "stop"]:
        console.print("[bold red]Stopping program. Goodbye![/bold red]")
        break

    if not user_prompt.strip():
        continue

    # 2. Search Database
    query_vector = encoder.encode(user_prompt).tolist()
    try:
        hits = qdrant.search(collection_name="nist_controls", query_vector=query_vector, limit=3)
    except AttributeError:
        # Fallback for different versions
        hits = qdrant.query_points(collection_name="nist_controls", query=query_vector, limit=3).points

    # 3. Prepare Context
    context_text = "\n".join([
        f"- ID: {hit.payload.get('identifier')} | Title: {hit.payload.get('name')} | Text: {hit.payload.get('control_text')}"
        for hit in hits
    ])

    # Display Results Table
    from rich.table import Table
    table = Table(title="Relevant NIST 800-53 Controls", show_lines=True)
    table.add_column("Control ID", style="bright_red")
    table.add_column("Title", style="green")
    table.add_column("Snippet", style="yellow")
    table.add_column("Score", style="#a6accd")

    for hit in hits:
        snippet = str(hit.payload.get("control_text", ""))[:150] + "..."
        table.add_row(
            str(hit.payload.get("identifier", "N/A")),
            str(hit.payload.get("name", "N/A")),
            snippet,
            f"{hit.score:.4f}"
        )
    console.print(table)

    # 4. Stream Answer
    messages = [
        {"role": "system", "content": "You are a NIST SP 800-53 expert. Answer the user based strictly on the context provided."},
        {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {user_prompt}\n\nAnswer:"}
    ]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)

    # Initialize Streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(inputs, streamer=streamer, max_new_tokens=400)

    # Run generation in a separate thread so we can print while it works
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    console.print("\n[bold green]NIST Recommendation:[/bold green]")

    # Print tokens as they arrive
    generated_text = ""
    for new_text in streamer:
        print(new_text, end="", flush=True)
        generated_text += new_text
    print() # Newline at end

✅ System Ready! Type 'exit' or 'quit' to stop.

Your Question:

>>> tell me about firewalls


                                           Relevant NIST 800-53 Controls                                           
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Control ID ┃ Title                                       ┃ Snippet                                     ┃ Score  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ SC-7(26)   │ Boundary Protection | Classified National   │ Prohibit the direct connection of a         │ 0.5060 │
│            │ Security System Connections                 │ classified national security system to an   │        │
│            │                                             │ external network without the use of         │        │
│            │                                             │ [Assignment: organization-defined bo...     │        │
├────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼────────┤
│ SC-7(25)   │ Boundary Protection | Unclassified National │ Prohibit the direct connection of           │ 0.4977 │
│            │ Security System Connections                 │ [Assignment: organization-defined           │        │
│            │                                             │ unclassified national security system] to   │        │
│            │                                             │ an external network without the use of [... │        │
├────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼────────┤
│ SC-7(27)   │ Boundary Protection | Unclassified          │ Prohibit the direct connection of           │ 0.4939 │
│            │ Non-national Security System Connections    │ [Assignment: organization-defined           │        │
│            │                                             │ unclassified non-national security system]  │        │
│            │                                             │ to an external network without the use ...  │        │
└────────────┴─────────────────────────────────────────────┴─────────────────────────────────────────────┴────────┘

NIST Recommendation:

...

NIST Recommendation:

Based on the given context, there is no specific mention of phishing directly. However, phishing can be considered a form of suspicious communication and anomalous behavior that organizations should be aware of. The context provides guidance on literacy training and awareness regarding suspicious communications and anomalous system behavior. 

For phishing specifically, the literacy training should cover recognizing phishing attempts. This would involve teaching employees how to identify suspicious emails or links that mimic legitimate sources but are actually designed to steal information. The training should also include steps on how to report such incidents and what actions to take if they suspect they have been phished.

Additionally, the context mentions providing literacy training on the advanced persistent threat (APT). Phishing is often used as one of the tactics employed by APTs, so this training indirectly touches upon phishing through the broader topic of APTs. However, dire

Your Question:

>>> exit


Stopping program. Goodbye!